In [68]:
import sys  # Para detectar ejecución empaquetada (frozen) y resolver rutas
import os   # Para construir rutas relativas al proyecto
import pandas as pd  # Principal herramienta para manipulación tabular
import numpy as np   # Utilidades numéricas/condicionales (np.select, np.where)
from openpyxl import load_workbook  # Leer valores dinámicos desde archivos Excel

from io import StringIO

import re


In [87]:
remittance = pd.read_excel("Remittance_Oxxo.xlsx", skiprows=11)

In [88]:
if remittance.columns[0].startswith("Unnamed"):
    remittance = remittance.iloc[:, 1:]

In [89]:
remittance = remittance.dropna(subset=["Unidad Operativa","Número Factura"])

In [93]:
remittance_1.head(1000)

,Unidad Operativa,Número Pago,Número Factura,Fecha Factura,Importe Factura,Importe Pagado,Impuesto,Retención
0,OXXOCOPRINC,8000178496,NCMI10336847,2025-07-28 00:00:00,-542243,-542243,-74120,0
1,OXXOCOPRINC,8000178496,NCM11207409,2025-07-29 00:00:00,-243600,-243600,0,0
2,OXXOCOPRINC,8000178496,NCM11207369,2025-07-29 00:00:00,-49924,-49924,0,0
3,OXXOCOPRINC,8000178496,NCM11207376,2025-07-29 00:00:00,-98913,-98913,0,0
4,OXXOCOPRINC,8000178496,NCM11207412,2025-07-29 00:00:00,-39099,-39099,0,0
5,OXXOCOPRINC,8000178496,NCM11207411,2025-07-29 00:00:00,-596512,-596512,0,0
6,OXXOCOPRINC,8000178496,NCM11207762,2025-07-29 00:00:00,-13781,-13781,0,0
7,OXXOCOPRINC,8000178496,NCM11207444,2025-07-29 00:00:00,-6559,-6559,0,0
8,OXXOCOPRINC,8000178496,PMP1270986,2025-08-06 00:00:00,7407755,6787530,1182751,0
9,OXXOCOPRINC,8000178496,PMP1270985,2025-08-06 00:00:00,68068389,68068389,10868062,0


In [98]:
df = remittance.copy()

def row_to_cells(row):
    vals = [str(x).strip() for x in row if str(x).strip() != ""]
    if len(vals) == 0:
        return []
    if len(vals) == 1:
        s = vals[0]
        if "\t" in s:
            parts = re.split(r'\t+', s)
        else:
            parts = re.split(r'\s{2,}', s)
        parts = [p.strip() for p in parts if p.strip() != ""]
        return parts
    else:
        return vals

rows = []
for _, r in df.iterrows():
    rows.append(row_to_cells(r.tolist()))

header_positions = [i for i, cells in enumerate(rows) if any("unidad operativa" in str(c).lower() for c in cells)]

# 💡 si el DataFrame empieza con datos, aseguramos que el inicio esté incluido
if 0 not in header_positions:
    header_positions = [0] + header_positions

header_positions.append(len(rows))

default_header = ["Unidad Operativa", "Número Pago", "Número Factura", "Fecha Factura",
                  "Importe Factura", "Importe Pagado", "Impuesto", "Retención"]

tables = []
for i in range(len(header_positions)-1):
    start = header_positions[i]
    # si la fila de inicio es un encabezado detectado, salta esa fila
    if any("unidad operativa" in str(c).lower() for c in rows[start]):
        start += 1
    end = header_positions[i+1]

    block = rows[start:end]
    if len(block) == 0:
        continue

    header_row = rows[header_positions[i]] if any("unidad operativa" in str(c).lower() for c in rows[header_positions[i]]) else default_header
    cols = header_row if len(header_row) >= 3 else default_header

    norm_block = []
    for r_cells in block:
        if len(r_cells) == 0:
            continue
        if len(r_cells) >= len(cols):
            row_fixed = r_cells[:len(cols)]
        else:
            row_fixed = r_cells + [None] * (len(cols) - len(r_cells))
        norm_block.append(row_fixed)

    subdf = pd.DataFrame(norm_block, columns=cols)
    tables.append(subdf.reset_index(drop=True))

for n, tdf in enumerate(tables, start=1):
    globals()[f"remittance_{n}"] = tdf
 #   print(f"✅ remittance_{n} creado con {len(tdf)} filas.")

#print(f"\nTotal tablas creadas: {len(tables)}")

In [99]:

def transform_remittance(df: pd.DataFrame) -> pd.DataFrame:

    # 1️⃣ Selección de columnas
    cols_to_keep = ["Número Factura", "Importe Factura", "Número Pago"]
    df = df[cols_to_keep].copy()

    # 2️⃣ Renombrar columnas
    df = df.rename(columns={
        "Número Factura": "Referencia / Factura",
        "Importe Factura": "Importe de Remittance"
    })

    # 3️⃣ Convertir importes a numérico y redondear
    df["Importe de Remittance"] = pd.to_numeric(df["Importe de Remittance"], errors="coerce").round(2)
    df["Impuesto"] = pd.to_numeric(df["Impuesto"], errors="coerce").round(2)

    # 4️⃣ Limpiar caracteres invisibles y espacios en 'Referencia / Factura'
    df["Referencia / Factura"] = df["Referencia / Factura"] \
        .str.replace(r"[\u202A-\u202E\u200E\u200F]", "", regex=True) \
        .str.strip()

    # 5️⃣ Crear columna 'Tipo de Documento'
    conds = [
        df["Referencia / Factura"].str.startswith("PMP", na=False) & (df["Importe de Remittance"] > 0),
        ~df["Referencia / Factura"].str.startswith("PMP", na=False) & (df["Importe de Remittance"] > 0),
        df["Importe de Remittance"] < 0
    ]
    choices = ["Factura", "Nota Debito", "Descuentos Clientes"]
    df["Tipo de Documento"] = np.select(conds, choices, default="")

    # Limpiar índice
    df.reset_index(drop=True, inplace=True)
    
    return df


In [100]:
# Lista de nombres de DataFrames a transformar
dfs_names = ["remittance_1", "remittance_2", "remittance_3", "remittance_4"]

for name in dfs_names:
    # Obtenemos el DataFrame por nombre, transformamos y reasignamos
    globals()[name] = transform_remittance(globals()[name])
#    print(f"{name} transformado. Columnas: {globals()[name].columns.tolist()}, Filas: {len(globals()[name])}")


In [107]:
def asignar_cards(remittance: pd.DataFrame) -> pd.DataFrame:
    # 1️⃣ Asegurar que existan las columnas
    if "Descuento" not in remittance.columns:
        remittance["Descuento"] = ""
    if "Motivo del descuento" not in remittance.columns:
        remittance["Motivo del descuento"] = ""

    conds_desc = [
        remittance["Referencia / Factura"].str.startswith("NCM", na=False),
        remittance["Referencia / Factura"].str.startswith("74", na=False),
        remittance["Referencia / Factura"].str.startswith("90", na=False),
        remittance["Referencia / Factura"].str.startswith("91", na=False),
        remittance["Referencia / Factura"].str.startswith("92", na=False),
        remittance["Referencia / Factura"].str.startswith("395", na=False),
        remittance["Referencia / Factura"].str.startswith("396", na=False),
        remittance["Referencia / Factura"].str.startswith("397", na=False)
    ]

    descuentos = [
        "CONVENIOS", "DSCT PROMOCIONAL", "DSCT PROMOCIONAL", "FACT PROVEEDOR",
        "CONVENIOS", "DPP", "RECHAZOS", "DSCT AVERIAS"
    ]

    motivos = ["657", "987", "987", "CSB", "657", "206", "551", "522"]

    remittance["Descuento"] = np.select(conds_desc, descuentos, default=remittance["Descuento"])
    remittance["Motivo del descuento"] = np.select(conds_desc, motivos, default=remittance["Motivo del descuento"])

    return remittance


In [108]:
for name in dfs_names:
    globals()[name] = asignar_cards(globals()[name])
#    print(f"{name} actualizado: columnas 'Descuento' y 'Motivo del descuento' añadidas")


In [109]:
remittance_1.head(100)

,Referencia / Factura,Importe de Remittance,Número Pago,Impuesto,Tipo de Documento,Descuento,Motivo del descuento
0,NCMI10336847,-542243,8000178496,-74120,Descuentos Clientes,CONVENIOS,657
1,NCM11207409,-243600,8000178496,0,Descuentos Clientes,CONVENIOS,657
2,NCM11207369,-49924,8000178496,0,Descuentos Clientes,CONVENIOS,657
3,NCM11207376,-98913,8000178496,0,Descuentos Clientes,CONVENIOS,657
4,NCM11207412,-39099,8000178496,0,Descuentos Clientes,CONVENIOS,657
5,NCM11207411,-596512,8000178496,0,Descuentos Clientes,CONVENIOS,657
6,NCM11207762,-13781,8000178496,0,Descuentos Clientes,CONVENIOS,657
7,NCM11207444,-6559,8000178496,0,Descuentos Clientes,CONVENIOS,657
8,PMP1270986,7407755,8000178496,1182751,Factura,,
9,PMP1270985,68068389,8000178496,10868062,Factura,,
